# Day 5 — LLMOps A/B Test Dashboard
**Goal:** Streamlit dashboard comparing **vanilla Mistral-7B-Instruct-v0.3 (base)** vs **NyayaGPT (fine-tuned)**, both running at Q4_K_M quantization through llama.cpp.
Using identical quantization isolates fine-tuning as the only variable — every quality delta is attributable to training on the 1,521 Indian legal Q&A pairs.

Each request is logged to MLflow under `nyayagpt-ab-test`, along with the randomly assigned variant label used in session stats.

**Architecture:**
```
Browser → Streamlit UI → ab_generate() → [Mistral Q4_K_M (base) + NyayaGPT Q4_K_M (fine-tuned)]
                              ↓
                       MLflow (logs latencies + variant tag per request)
```

In [1]:
import sys, os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
os.chdir(PROJECT_ROOT)

# Preflight only — avoid mutating the environment from the notebook.
import streamlit, mlflow
print(f'✓ streamlit {streamlit.__version__}')
print(f'✓ mlflow    {mlflow.__version__}')

✓ streamlit 1.56.0
✓ mlflow    3.11.1


In [2]:
# Check prerequisites
from nyaya_pipeline import config
assert config.ADAPTER_DIR.exists(), f'Adapter not found. Run Day 2 first.'
print(f'✓ Adapter: {config.ADAPTER_DIR}')

base_gguf = Path('/mnt/f/NyayaGPT-scratch/mistral-base-q4km.gguf')
ft_gguf   = config.ADAPTER_DIR / 'nyayagpt-q4km.gguf'
assert base_gguf.exists(), (
    f'Base Mistral Q4_K_M GGUF missing at {base_gguf}. '
    f'Run: python scripts/build_base_mistral_gguf.py'
)
assert ft_gguf.exists(), 'NyayaGPT Q4_K_M GGUF missing. Run Day 4 first.'
print(f'✓ Base Mistral GGUF: {base_gguf}  ({base_gguf.stat().st_size / 1e9:.2f} GB)')
print(f'✓ NyayaGPT GGUF:     {ft_gguf}  ({ft_gguf.stat().st_size / 1e9:.2f} GB)')

dashboard_path = PROJECT_ROOT / 'src' / 'nyaya_pipeline' / 'dashboard.py'
assert dashboard_path.exists(), f'dashboard.py not found at {dashboard_path}'
print(f'✓ Dashboard: {dashboard_path}')

✓ Adapter: /home/ubuntu/Fine-tuning/NyayaGPT/adapters
✓ Base Mistral GGUF: /mnt/f/NyayaGPT-scratch/mistral-base-q4km.gguf  (4.37 GB)
✓ NyayaGPT GGUF:     /home/ubuntu/Fine-tuning/NyayaGPT/adapters/nyayagpt-q4km.gguf  (4.37 GB)
✓ Dashboard: /home/ubuntu/Fine-tuning/NyayaGPT/src/nyaya_pipeline/dashboard.py


## Preview the A/B routing logic
(Without launching Streamlit)

In [3]:
# Reload the inference module so this notebook picks up the latest GGUF-based A/B path
# Reload the inference module so this notebook picks up the latest GGUF-based A/B path
import importlib, inspect
import nyaya_pipeline.infer as infer_mod

infer_mod = importlib.reload(infer_mod)
ab_generate = infer_mod.ab_generate
print(inspect.getsource(ab_generate))

def ab_generate(
    question: str,
    base_model: Optional[str] = None,
    finetuned_model: Optional[str] = None,
    adapter_dir: Optional[Path] = None,
    max_new_tokens: int = 256,
) -> Tuple[str, str, str, float, float]:
    """
    A/B routing comparing vanilla Mistral against NyayaGPT, both at Q4_K_M GGUF.

    Uses identical quantization for both sides so the only variable is
    fine-tuning itself — every quality delta is attributable to training on
    Indian legal data:
      - "base"      -> vanilla Mistral-7B-Instruct-v0.3 Q4_K_M GGUF
      - "finetuned" -> NyayaGPT Q4_K_M GGUF

    Returns:
        (assigned_variant, base_response, finetuned_response, base_latency_ms, ft_latency_ms)
    """
    import mlflow

    del base_model, finetuned_model, adapter_dir  # legacy args retained for notebook/API stability
    variant         = random.choice(["base", "finetuned"])

    # Always generate both for side-by-side display. We avoid the HF/Unsloth
    # path here because Bla

In [4]:
# Test the side-by-side GGUF generation path manually (without Streamlit)
# This loads the FP16 and INT4 NyayaGPT GGUF models and compares them.
question = 'What is the maximum punishment under Section 302 IPC?'

print('Testing A/B routing …')
variant, base_resp, ft_resp, base_ms, ft_ms = ab_generate(
    question=question,
    max_new_tokens=128,
)

print(f'Assigned variant: {variant}')
print(f'\nNyayaGPT FP16 ({base_ms:.0f} ms):')
print(f'  {base_resp[:200]}')
print(f'\nNyayaGPT INT4 ({ft_ms:.0f} ms):')
print(f'  {ft_resp[:200]}')

Testing A/B routing …
2026-04-25 11:32:52,327 [INFO] nyaya_pipeline.infer: Loading GGUF model: /mnt/f/NyayaGPT-scratch/mistral-base-q4km.gguf
2026-04-25 11:32:52,329 [INFO] nyaya_pipeline.infer: (silent for ~20s while weights transfer to GPU)


llama_context: n_ctx_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


2026-04-25 11:33:09,873 [INFO] nyaya_pipeline.infer: Loading GGUF model: /home/ubuntu/Fine-tuning/NyayaGPT/adapters/nyayagpt-q4km.gguf
2026-04-25 11:33:09,875 [INFO] nyaya_pipeline.infer: (silent for ~20s while weights transfer to GPU)


llama_context: n_ctx_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


Assigned variant: base

NyayaGPT FP16 (17534 ms):
  Section 302 of the Indian Penal Code (IPC) pertains to murder, which is defined as the unlawful killing of a person with intent after premeditation. The maximum punishment for this offense is death or

NyayaGPT INT4 (15933 ms):
  The maximum punishment for an offence punishable under Section 302 of the Indian Penal Code (IPC) is death penalty or life imprisonment, as prescribed by the statute.


/home/ubuntu/Fine-tuning/.venv/lib/python3.10/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


## Launch Streamlit Dashboard

In [7]:
# Start MLflow UI if port 5000 is not already in use
import socket, subprocess, sys
from nyaya_pipeline import config

def _port_open(host: str, port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.5)
        return s.connect_ex((host, port)) == 0

if _port_open('127.0.0.1', 5000):
    print('MLflow UI already running → http://localhost:5000')
else:
    mlflow_proc = subprocess.Popen(
        [sys.executable, '-m', 'mlflow', 'ui',
         '--backend-store-uri', config.MLFLOW_URI,
         '--port', '5000'],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    print('MLflow UI started → http://localhost:5000')

MLflow UI already running → http://localhost:5000


In [6]:
# Launch Streamlit dashboard
# This will block the notebook cell — open in a terminal instead for a non-blocking run:
#   cd /home/ubuntu/Fine-tuning/NyayaGPT
#   streamlit run src/nyaya_pipeline/dashboard.py --server.port 8501

print('To launch the dashboard, run this in a terminal:')
print()
print(f'  cd {PROJECT_ROOT}')
print(f'  streamlit run src/nyaya_pipeline/dashboard.py --server.port 8501')
print()
print('Then open: http://localhost:8501')

# Uncomment to launch from notebook (blocks until stopped):
# subprocess.run([sys.executable, '-m', 'streamlit', 'run',
#                 str(dashboard_path), '--server.port', '8501'])

To launch the dashboard, run this in a terminal:

  cd /home/ubuntu/Fine-tuning/NyayaGPT
  streamlit run src/nyaya_pipeline/dashboard.py --server.port 8501

Then open: http://localhost:8501


## Dashboard UI Preview
```
┌──────────────────────────────────────────────────────────────┐
│  ⚖️  NyayaGPT — A/B Test Dashboard                           │
│  Base Mistral vs NyayaGPT (fine-tuned) — both Q4_K_M         │
│  ─────────────────────────────────────────                   │
│  Question: [_______________________________] [Send ⚡]       │
│                                                              │
│  🔵 Base Mistral (vanilla)   🟢 NyayaGPT (fine-tuned)        │
│  ┌────────────────────┐   ┌─────────────────────────┐        │
│  │ Generic response…  │   │ IPC §302 cited, legal…  │        │
│  │ Latency: 4200 ms   │   │ Latency: 4100 ms        │        │
│  └────────────────────┘   └─────────────────────────┘        │
│                                                              │
│  Total: 7 | NyayaGPT served: 4/7 | Avg delta: -100 ms        │
└──────────────────────────────────────────────────────────────┘
```

## Day 5 Run Checklist

**MLflow A/B logs:** `mlruns/` → experiment `nyayagpt-ab-test`

**Then:** Run `06_hf_hub_deployment.ipynb`